# Task 2: Dense retrieval

Two pretrained bi-encoders, used zero-shot:

- **general**: `BAAI/bge-base-en-v1.5` (109M params, trained on general web / QA pairs)
- **scientific**: `ncbi/MedCPT` (PubMedBERT based, trained on PubMed search logs). It has a separate query encoder and article encoder, 109M each.

Every abstract is embedded once (the "index"), each claim is embedded at query time, and the ranking is just the dot product between the two. With 5183 abstracts an exact brute-force search is instant, so no ANN index is used.

Needs the outputs of notebook 1 (the `NLP_Inter_IIT.zip` from its last cell). Use a GPU runtime (about 2-3 min on a T4, much slower on CPU).

In [1]:
# torch is assumed to be installed already (it is on colab)
%pip install -q transformers PyStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 8.9 MB/s eta 0:00:00


In [2]:
import json, math, re, sys, time, zipfile
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import Stemmer
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

# project folder (same layout as NLP_Inter_IIT.zip):
#   NLP_Inter_IIT/
#     data/  check_format.py  results_template.json
#     predictions/  <- run files for the submission
#     work/         <- cached scores passed from one notebook to the next (not submitted)
#
# Every colab notebook runs on its own machine, so files don't carry over by themselves.
# The last cell of each notebook packs the folder into NLP_Inter_IIT.zip and this cell unpacks it:
# when asked, upload the zip from the previous notebook (for notebook 1, the original zip).
# Not needed when running locally from NLP_Inter_IIT/ or NLP_Inter_IIT/code/.

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd() / "NLP_Inter_IIT"]:
        if (p / "data" / "corpus.jsonl").exists():
            return p

if find_root() is None:
    zips = list(Path.cwd().glob("*.zip"))
    if not zips:
        from google.colab import files
        zips = [Path(name) for name in files.upload()]
    newest = max(zips, key=lambda z: z.stat().st_mtime)
    zipfile.ZipFile(newest).extractall(".")
    print("unpacked", newest.name)

ROOT = find_root()
assert ROOT is not None, "data/corpus.jsonl not found, upload NLP_Inter_IIT.zip"
DATA, PRED, WORK = ROOT / "data", ROOT / "predictions", ROOT / "work"
assert (WORK / "scores_bm25_val.npy").exists(), "work/ is missing: upload the NLP_Inter_IIT.zip that notebook 1 downloaded in its last cell"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "none (cpu)"
print(device, "|", gpu_name)

Saving NLP_Inter_IIT.zip to NLP_Inter_IIT.zip
unpacked NLP_Inter_IIT.zip
cuda | Tesla T4


## Data and helpers (same as notebook 1)

In [3]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

corpus = sorted(read_jsonl(DATA / "corpus.jsonl"), key=lambda d: d["doc_id"])
doc_ids = np.array([d["doc_id"] for d in corpus])
claims = {s: sorted(read_jsonl(DATA / f"{s}_claims.jsonl"), key=lambda c: c["id"]) for s in ["train", "val", "eval"]}

TOP_K = 100

def gold_docs(split):
    return {c["id"]: {e["doc_id"] for e in c["evidence"]} for c in claims[split]}

def make_run(scores, split, k=TOP_K):
    order = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    run = {}
    for i, c in enumerate(claims[split]):
        run[c["id"]] = [(int(doc_ids[j]), float(scores[i, j])) for j in order[i]]
    return run

def claim_metrics(run, gold):
    rows = {}
    for cid, g in gold.items():
        if len(g) == 0:
            continue
        ranked = [d for d, _ in run[cid]]
        dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(ranked[:10]) if d in g)
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(g), 10)))
        rr = 0.0
        for r, d in enumerate(ranked):
            if d in g:
                rr = 1 / (r + 1)
                break
        rows[cid] = {"ndcg10": dcg / idcg, "recall100": len(g & set(ranked)) / len(g), "mrr": rr}
    return pd.DataFrame.from_dict(rows, orient="index")

def evaluate(run, gold):
    return claim_metrics(run, gold).mean().round(4).to_dict()

def write_trec(path, run, name):
    with open(path, "w") as f:
        for cid in sorted(run):
            for rank, (did, score) in enumerate(run[cid], start=1):
                f.write(f"{cid} Q0 {did} {rank} {score:.6f} {name}\n")

## Models

Both models use the [CLS] vector as the embedding, the differences are in how they expect the input:

| | bge-base-en-v1.5 | MedCPT |
|---|---|---|
| encoders | one shared model | query encoder + article encoder |
| query input | instruction prefix + claim | claim |
| document input | title + abstract as one string | (title, abstract) as a sentence pair |
| similarity | cosine (embeddings are L2-normalised) | dot product |

In [4]:
MODELS = {
    "general": {
        "query_model": "BAAI/bge-base-en-v1.5",
        "doc_model": "BAAI/bge-base-en-v1.5",
        # bge v1.5 wants this instruction in front of short queries (not in front of documents)
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "normalize": True,
        "title_pair": False,
    },
    "scientific": {
        "query_model": "ncbi/MedCPT-Query-Encoder",
        "doc_model": "ncbi/MedCPT-Article-Encoder",
        "query_prefix": "",
        "normalize": False,
        "title_pair": True,
    },
}

QUERY_MAX_LEN = 64     # claims are ~12 words
DOC_MAX_LEN = 512      # abstracts are ~215 words on average, so most fit without truncation
BATCH_SIZE = 64

In [5]:
def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModel.from_pretrained(name).to(device).eval()
    return tok, model

def n_params(model):
    return sum(p.numel() for p in model.parameters())

def weights_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6

@torch.no_grad()
def embed(tok, model, texts, max_len, text_pairs=None, normalize=False, batch_size=BATCH_SIZE, progress=False):
    # returns a (len(texts), dim) float32 array of [CLS] embeddings
    out = []
    steps = range(0, len(texts), batch_size)
    for i in tqdm(steps) if progress else steps:
        second = text_pairs[i:i + batch_size] if text_pairs is not None else None
        batch = tok(texts[i:i + batch_size], second, padding=True, truncation=True,
                    max_length=max_len, return_tensors="pt").to(device)
        # fp16 on gpu, roughly 2x faster and the rankings don't change
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            cls = model(**batch).last_hidden_state[:, 0]
        cls = cls.float()
        if normalize:
            cls = F.normalize(cls, dim=-1)
        out.append(cls.cpu().numpy())
    return np.vstack(out)

def sync():
    # gpu calls are async, wait for them before reading the clock
    if device == "cuda":
        torch.cuda.synchronize()

## Index + retrieve

For each model:
1. embed all abstracts and time it (**index time**)
2. embed the claims of each split in batches and save the full score matrix (needed for the hybrid in task 3 and for hard negatives in task 4)
3. measure **query latency** one claim at a time on val: embed the claim + score against all abstracts + take the top 100
4. **memory**: size of the embedding matrix + weights of the query encoder, i.e. what has to stay loaded to answer queries

In [6]:
titles = [d["title"] for d in corpus]
abstracts = [" ".join(d["abstract"]) for d in corpus]

def run_dense(name):
    cfg = MODELS[name]
    q_tok, q_model = load(cfg["query_model"])
    if cfg["doc_model"] == cfg["query_model"]:
        d_tok, d_model = q_tok, q_model
    else:
        d_tok, d_model = load(cfg["doc_model"])
    print(f"{name}: query encoder {n_params(q_model) / 1e6:.0f}M params, doc encoder {n_params(d_model) / 1e6:.0f}M params")
    assert max(n_params(q_model), n_params(d_model)) <= 400e6      # rule: at most 400M params per model

    # 1. index
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sync(); t0 = time.perf_counter()
    if cfg["title_pair"]:
        doc_emb = embed(d_tok, d_model, titles, DOC_MAX_LEN, text_pairs=abstracts, normalize=cfg["normalize"], progress=True)
    else:
        doc_emb = embed(d_tok, d_model, [t + " " + a for t, a in zip(titles, abstracts)], DOC_MAX_LEN,
                        normalize=cfg["normalize"], progress=True)
    sync(); index_time = time.perf_counter() - t0
    peak_gpu_mb = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else None

    # 2. score matrices for all splits
    scores = {}
    for split in ["train", "val", "eval"]:
        queries = [cfg["query_prefix"] + c["claim"] for c in claims[split]]
        q_emb = embed(q_tok, q_model, queries, QUERY_MAX_LEN, normalize=cfg["normalize"])
        scores[split] = q_emb @ doc_emb.T
        np.save(WORK / f"scores_{name}_{split}.npy", scores[split])

    # 3. latency per query (batch size 1), after a few warm-up queries
    val_queries = [cfg["query_prefix"] + c["claim"] for c in claims["val"]]
    for q in val_queries[:5]:
        embed(q_tok, q_model, [q], QUERY_MAX_LEN, normalize=cfg["normalize"])
    times = []
    for q in val_queries:
        sync(); t0 = time.perf_counter()
        q_emb = embed(q_tok, q_model, [q], QUERY_MAX_LEN, normalize=cfg["normalize"])
        s = (q_emb @ doc_emb.T)[0]
        top = np.argpartition(-s, TOP_K)[:TOP_K]
        top = top[np.argsort(-s[top])]
        sync(); times.append((time.perf_counter() - t0) * 1000)

    # 4. memory
    index_mb = doc_emb.nbytes / 1e6
    model_mb = weights_mb(q_model)

    stats = {
        "query_model": cfg["query_model"], "doc_model": cfg["doc_model"],
        "dim": int(doc_emb.shape[1]),
        "index_time_sec": round(index_time, 2),
        "query_latency_ms": round(float(np.mean(times)), 2),
        "query_latency_ms_median": round(float(np.median(times)), 2),
        "index_mb": round(index_mb, 1),
        "query_encoder_mb": round(model_mb, 1),
        "memory_mb": round(index_mb + model_mb, 1),
        "peak_gpu_mb_while_indexing": None if peak_gpu_mb is None else round(peak_gpu_mb, 1),
    }

    del q_model, d_model
    if device == "cuda":
        torch.cuda.empty_cache()
    return scores, stats

In [7]:
scores, stats = {}, {}
for name in ["general", "scientific"]:
    scores[name], stats[name] = run_dense(name)

pd.DataFrame(stats).T[["dim", "index_time_sec", "query_latency_ms", "index_mb", "query_encoder_mb", "memory_mb"]]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

scientific: query encoder 109M params, doc encoder 109M params


  0%|          | 0/81 [00:00<?, ?it/s]

,dim,index_time_sec,query_latency_ms,index_mb,query_encoder_mb,memory_mb
general,768,49.12,33.52,15.9,437.9,453.9
scientific,768,42.59,46.67,15.9,437.9,453.9


## Results on val

In [8]:
scores["bm25"] = {s: np.load(WORK / f"scores_bm25_{s}.npy") for s in ["train", "val", "eval"]}

table = {}
for name in ["bm25", "general", "scientific"]:
    for split in ["val", "train"]:
        m = evaluate(make_run(scores[name][split], split), gold_docs(split))
        table[(name, split)] = m
results = pd.DataFrame(table).T
results

ndcg10  recall100     mrr
bm25       val    0.8735     0.9811  0.8546
           train  0.8519     0.9881  0.8310
general    val    0.8976     1.0000  0.8877
           train  0.9142     0.9975  0.8934
scientific val    0.8509     1.0000  0.8127
           train  0.8929     0.9975  0.8689

In [9]:
def cand(model, pooling, query_prefix="", doc_prefix="", normalize=True, title_pair=False, doc_model=None):
    return {"query_model": model, "doc_model": doc_model or model, "pooling": pooling,
            "query_prefix": query_prefix, "doc_prefix": doc_prefix,
            "normalize": normalize, "title_pair": title_pair}

BGE_PREFIX = "Represent this sentence for searching relevant passages: "

CANDIDATES = {
    "general": {
        "bge-base-en-v1.5": cand("BAAI/bge-base-en-v1.5", "cls", query_prefix=BGE_PREFIX),      # already run above
        "bge-large-en-v1.5": cand("BAAI/bge-large-en-v1.5", "cls", query_prefix=BGE_PREFIX),    # 335M
        "e5-base-v2": cand("intfloat/e5-base-v2", "mean", query_prefix="query: ", doc_prefix="passage: "),
        "gte-base": cand("thenlper/gte-base", "mean"),
        "all-mpnet-base-v2": cand("sentence-transformers/all-mpnet-base-v2", "mean"),
    },
    "scientific": {
        "MedCPT": cand("ncbi/MedCPT-Query-Encoder", "cls", normalize=False, title_pair=True,
                       doc_model="ncbi/MedCPT-Article-Encoder"),                                 # already run above
        "S-PubMedBert-MS-MARCO": cand("pritamdeka/S-PubMedBert-MS-MARCO", "mean"),
        "pubmedbert-base-embeddings": cand("NeuML/pubmedbert-base-embeddings", "mean"),
        "BioLORD-2023": cand("FremyCompany/BioLORD-2023", "mean"),
    },
}

@torch.no_grad()
def embed_pooled(tok, model, texts, max_len, pooling, text_pairs=None, normalize=True, batch_size=BATCH_SIZE):
    # same as embed() above, but pooling can be [CLS] or the mean over all real tokens
    out = []
    for i in range(0, len(texts), batch_size):
        second = text_pairs[i:i + batch_size] if text_pairs is not None else None
        batch = tok(texts[i:i + batch_size], second, padding=True, truncation=True,
                    max_length=max_len, return_tensors="pt").to(device)
        with torch.autocast(device_type=device, enabled=(device == "cuda")):
            hidden = model(**batch).last_hidden_state
        hidden = hidden.float()
        if pooling == "cls":
            emb = hidden[:, 0]
        else:
            mask = batch["attention_mask"].unsqueeze(-1).float()      # padding tokens don't count
            emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        if normalize:
            emb = F.normalize(emb, dim=-1)
        out.append(emb.cpu().numpy())
    return np.vstack(out)

def run_candidate(cfg):
    # same steps and measurements as run_dense(), nothing is written to disk here
    q_tok, q_model = load(cfg["query_model"])
    d_tok, d_model = (q_tok, q_model) if cfg["doc_model"] == cfg["query_model"] else load(cfg["doc_model"])
    assert max(n_params(q_model), n_params(d_model)) <= 400e6

    def embed_q(texts):
        return embed_pooled(q_tok, q_model, [cfg["query_prefix"] + t for t in texts], QUERY_MAX_LEN,
                            cfg["pooling"], normalize=cfg["normalize"])

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    sync(); t0 = time.perf_counter()
    if cfg["title_pair"]:
        doc_emb = embed_pooled(d_tok, d_model, titles, DOC_MAX_LEN, cfg["pooling"], text_pairs=abstracts,
                               normalize=cfg["normalize"])
    else:
        doc_emb = embed_pooled(d_tok, d_model, [cfg["doc_prefix"] + t + " " + a for t, a in zip(titles, abstracts)],
                               DOC_MAX_LEN, cfg["pooling"], normalize=cfg["normalize"])
    sync(); index_time = time.perf_counter() - t0
    peak_gpu_mb = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else None

    out = {split: embed_q([c["claim"] for c in claims[split]]) @ doc_emb.T for split in ["train", "val", "eval"]}

    val_claims = [c["claim"] for c in claims["val"]]
    for q in val_claims[:5]:
        embed_q([q])
    times = []
    for q in val_claims:
        sync(); t0 = time.perf_counter()
        s = (embed_q([q]) @ doc_emb.T)[0]
        top = np.argpartition(-s, TOP_K)[:TOP_K]
        top = top[np.argsort(-s[top])]
        sync(); times.append((time.perf_counter() - t0) * 1000)

    index_mb, model_mb = doc_emb.nbytes / 1e6, weights_mb(q_model)
    st = {
        "query_model": cfg["query_model"], "doc_model": cfg["doc_model"],
        "dim": int(doc_emb.shape[1]),
        "index_time_sec": round(index_time, 2),
        "query_latency_ms": round(float(np.mean(times)), 2),
        "query_latency_ms_median": round(float(np.median(times)), 2),
        "index_mb": round(index_mb, 1),
        "query_encoder_mb": round(model_mb, 1),
        "memory_mb": round(index_mb + model_mb, 1),
        "peak_gpu_mb_while_indexing": None if peak_gpu_mb is None else round(peak_gpu_mb, 1),
    }
    del q_model, d_model
    if device == "cuda":
        torch.cuda.empty_cache()
    return out, st

cand_scores, cand_stats = {}, {}

In [10]:
# about 1 min per model on a T4, bge-large takes longer. Results are cached in cand_scores,
# so rerunning this cell only computes the ones that are missing.
rows = []
for slot in ["general", "scientific"]:
    for short, cfg in CANDIDATES[slot].items():
        if short not in cand_scores:
            if cfg["query_model"] == MODELS[slot]["query_model"]:
                cand_scores[short], cand_stats[short] = scores[slot], stats[slot]      # reuse the run from above
            else:
                try:
                    cand_scores[short], cand_stats[short] = run_candidate(cfg)
                except Exception as e:
                    print(f"skipping {short}: {type(e).__name__}: {e}")
                    continue
        val = evaluate(make_run(cand_scores[short]["val"], "val"), gold_docs("val"))
        train = evaluate(make_run(cand_scores[short]["train"], "train"), gold_docs("train"))
        st = cand_stats[short]
        rows.append({"slot": slot, "model": short, "val_ndcg10": val["ndcg10"], "val_recall100": val["recall100"],
                     "val_mrr": val["mrr"], "train_ndcg10": train["ndcg10"], "dim": st["dim"],
                     "index_time_sec": st["index_time_sec"], "query_latency_ms": st["query_latency_ms"],
                     "memory_mb": st["memory_mb"]})
        print(slot, short, "val nDCG@10 =", val["ndcg10"])

comparison = pd.DataFrame(rows).set_index(["slot", "model"])
print("bm25 val nDCG@10 =", results.loc[("bm25", "val"), "ndcg10"])
comparison

general bge-base-en-v1.5 val nDCG@10 = 0.8976


config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

scientific BioLORD-2023 val nDCG@10 = 0.6989
bm25 val nDCG@10 = 0.8735


val_ndcg10  val_recall100  val_mrr  \
slot       model                                                            
general    bge-base-en-v1.5                0.8976         1.0000   0.8877   
           bge-large-en-v1.5               0.9328         1.0000   0.9191   
           e5-base-v2                      0.8818         0.9906   0.8548   
           gte-base                        0.8661         1.0000   0.8444   
           all-mpnet-base-v2               0.8240         0.9906   0.8032   
scientific MedCPT                          0.8509         1.0000   0.8127   
           S-PubMedBert-MS-MARCO           0.8402         1.0000   0.8196   
           pubmedbert-base-embeddings      0.7911         0.9906   0.7652   
           BioLORD-2023                    0.6989         0.9780   0.6803   

                                       train_ndcg10   dim  index_time_sec  \
slot       model                                                            
general    bge-base-en-v1.5                  0.9142   768           49.12   
           bge-large-en-v1.5                 0.9213  1024          117.43   
           e5-base-v2                        0.8728   768           45.29   
           gte-base                          0.8917   768           45.06   
           all-mpnet-base-v2                 0.8122   768           77.23   
scientific MedCPT                            0.8929   768           42.59   
           S-PubMedBert-MS-MARCO             0.8678   768           44.36   
           pubmedbert-base-embeddings        0.8086   768           43.45   
           BioLORD-2023                      0.6954   768           75.60   

                                       query_latency_ms  memory_mb  
slot       model                                                    
general    bge-base-en-v1.5                       33.52      453.9  
           bge-large-en-v1.5                      46.41     1361.8  
           e5-base-v2                             18.86      453.9  
           gte-base                               15.88      234.9  
           all-mpnet-base-v2                      26.22      453.9  
scientific MedCPT                                 46.67      453.9  
           S-PubMedBert-MS-MARCO                  25.01      453.9  
           pubmedbert-base-embeddings             18.87      453.9  
           BioLORD-2023                           20.91      453.9

In [11]:
# best val nDCG@10 per slot becomes the "general" / "scientific" system from here on:
# scores, timing stats and the cached score matrices in work/ are replaced
model_names = {}
for slot in ["general", "scientific"]:
    best = comparison.loc[slot]["val_ndcg10"].idxmax()        # on a tie the original pick stays
    cfg = CANDIDATES[slot][best]
    MODELS[slot] = cfg
    scores[slot], stats[slot] = cand_scores[best], cand_stats[best]
    for split in ["train", "val", "eval"]:
        np.save(WORK / f"scores_{slot}_{split}.npy", scores[slot][split])
    model_names[slot] = cfg["query_model"] if cfg["doc_model"] == cfg["query_model"] else cfg["query_model"] + " + " + cfg["doc_model"]
    print(f"{slot}: {best}  ({model_names[slot]})")

# same table as before, now with the selected models
table = {}
for name in ["bm25", "general", "scientific"]:
    for split in ["val", "train"]:
        table[(name, split)] = evaluate(make_run(scores[name][split], split), gold_docs(split))
results = pd.DataFrame(table).T
results

general: bge-large-en-v1.5  (BAAI/bge-large-en-v1.5)
scientific: MedCPT  (ncbi/MedCPT-Query-Encoder + ncbi/MedCPT-Article-Encoder)


ndcg10  recall100     mrr
bm25       val    0.8735     0.9811  0.8546
           train  0.8519     0.9881  0.8310
general    val    0.9328     1.0000  0.9191
           train  0.9213     0.9975  0.9012
scientific val    0.8509     1.0000  0.8127
           train  0.8929     0.9975  0.8689

(train numbers are only there as a second opinion, none of these models has seen the training claims yet. Model selection uses val.)

In [12]:
# run files
write_trec(PRED / "t2_general.trec", make_run(scores["general"]["eval"], "eval"), "general")
write_trec(PRED / "t2_scientific.trec", make_run(scores["scientific"]["eval"], "eval"), "scientific")
write_trec(PRED / "val_general.trec", make_run(scores["general"]["val"], "val"), "general")   # for report Q1

sys.path.append(str(ROOT))
from check_format import check_trec

all_docs = set(doc_ids.tolist())
eval_ids = {c["id"] for c in claims["eval"]}
val_ids = {c["id"] for c in claims["val"]}
check_trec(PRED / "t2_general.trec", eval_ids, all_docs)
check_trec(PRED / "t2_scientific.trec", eval_ids, all_docs)
check_trec(PRED / "val_general.trec", val_ids, all_docs)
print("3 run files written, all pass check_format")

3 run files written, all pass check_format


## BM25 vs general, claim by claim (report Q1)

Per-claim nDCG@10 on val for BM25 and the general model. The two claims with the biggest gap in each direction are the ones reported in `results.json`.

In [13]:
gold_val = gold_docs("val")
runs_val = {name: make_run(scores[name]["val"], "val") for name in ["bm25", "general"]}

cmp = pd.DataFrame({
    "bm25": claim_metrics(runs_val["bm25"], gold_val)["ndcg10"],
    "general": claim_metrics(runs_val["general"], gold_val)["ndcg10"],
})
cmp["diff"] = cmp["bm25"] - cmp["general"]

print("claims where bm25 is better:", (cmp["diff"] > 0).sum())
print("claims where general is better:", (cmp["diff"] < 0).sum())
print("same nDCG@10:", (cmp["diff"] == 0).sum())

# biggest gaps in each direction, ties broken by claim id so the choice is reproducible
cmp = cmp.sort_index()
bm25_wins = cmp[cmp["diff"] > 0].sort_values("diff", ascending=False, kind="stable").head(2).index.tolist()
general_wins = cmp[cmp["diff"] < 0].sort_values("diff", ascending=True, kind="stable").head(2).index.tolist()
if len(bm25_wins) < 2 or len(general_wins) < 2:
    print("WARNING: fewer than 2 claims in one direction, Q1 needs 2 + 2")
cmp.loc[bm25_wins + general_wins].round(3)

claims where bm25 is better: 9
claims where general is better: 18
same nDCG@10: 79


,bm25,general,diff
673,1.000,0.387,0.613
34,1.000,0.631,0.369
463,0.000,1.000,-1.000
541,0.204,0.920,-0.715


In [14]:
# look at the four claims: where is the gold abstract in each ranking, and which words do claim and abstract share?
stemmer = Stemmer.Stemmer("english")
GENERIC = set("a an and are as at be by for from in into is it its of on or that the their this to was were which with not no".split())

def terms(text):
    toks = re.findall(r"[a-z0-9]+", text.lower())
    return {stemmer.stemWord(t) for t in toks if t not in GENERIC}

docs_by_id = {d["doc_id"]: d for d in corpus}
claim_text = {c["id"]: c["claim"] for c in claims["val"]}

# document frequency of each term, to see which shared terms are rare (= what bm25 rewards)
df = Counter()
for d in corpus:
    df.update(terms(d["title"] + " " + " ".join(d["abstract"])))

def show(cid):
    print(f"[{cid}] {claim_text[cid]}")
    for gold_id in sorted(gold_val[cid]):
        doc = docs_by_id[gold_id]
        ranks = {}
        for name in ["bm25", "general"]:
            ranked = [d for d, _ in runs_val[name][cid]]
            ranks[name] = ranked.index(gold_id) + 1 if gold_id in ranked else ">100"
        c_terms = terms(claim_text[cid])
        d_terms = terms(doc["title"] + " " + " ".join(doc["abstract"]))
        shared = sorted(c_terms & d_terms, key=lambda t: df[t])
        print(f"    gold: {doc['title']}")
        print(f"    rank in bm25 = {ranks['bm25']}, rank in general = {ranks['general']}")
        print(f"    shared terms (rarest first, with doc freq): {[(t, df[t]) for t in shared]}")
        print(f"    claim terms not in the abstract: {sorted(c_terms - d_terms)}")
    print()

print("BM25 better\n")
for cid in bm25_wins:
    show(cid)
print("general better\n")
for cid in general_wins:
    show(cid)

BM25 better

[673] LDL cholesterol has a causal role in the development of cardiovascular disease.
    gold: LDL-cholesterol concentrations: a genome-wide association study
    rank in bm25 = 1, rank in general = 5
    shared terms (rarest first, with doc freq): [('ldl', 25), ('causal', 45), ('cholesterol', 92), ('cardiovascular', 181), ('diseas', 1169), ('role', 1281), ('develop', 1288), ('has', 1404)]
    claim terms not in the abstract: []

[34] A deficiency of folate increases blood levels of homocysteine.
    gold: Randomized trial of folic acid supplementation and serum homocysteine levels.
    rank in bm25 = 1, rank in general = 2
    shared terms (rarest first, with doc freq): [('folat', 16), ('homocystein', 24), ('blood', 424), ('level', 1099), ('increas', 1573)]
    claim terms not in the abstract: ['defici']

general better

[463] Genomic aberrations of metastases provide information for targeted therapy.
    gold: Evolution of metastasis revealed by mutational landscapes of

In [15]:
# save everything that goes into results.json later
log_path = WORK / "results_log.json"
results_log = json.load(open(log_path)) if log_path.exists() else {}
results_log["gpu_used"] = gpu_name
results_log["task2"] = {
    "general_model": model_names["general"],
    "scientific_model": model_names["scientific"],
    "val_ndcg10_general": results.loc[("general", "val"), "ndcg10"],
    "val_ndcg10_scientific": results.loc[("scientific", "val"), "ndcg10"],
    "val": {name: results.loc[(name, "val")].to_dict() for name in ["general", "scientific"]},
    "stats": {name: stats[name] for name in ["general", "scientific"]},
    # every model that was tried, so the choice can be justified in the report
    "candidates": json.loads(comparison.reset_index().to_json(orient="records")),
}
results_log["report_q1"] = {"bm25_wins": [int(c) for c in bm25_wins], "general_wins": [int(c) for c in general_wins]}
json.dump(results_log, open(log_path, "w"), indent=2)
print(results_log["task2"]["general_model"], "|", results_log["task2"]["scientific_model"])
results_log["report_q1"]

BAAI/bge-large-en-v1.5 | ncbi/MedCPT-Query-Encoder + ncbi/MedCPT-Article-Encoder


{'bm25_wins': [673, 34], 'general_wins': [463, 541]}

## Save for the next notebook

Packs data, predictions and work into `NLP_Inter_IIT.zip` and downloads it (colab only). Upload this zip in the first cell of the next notebook.

In [17]:
if "google.colab" in sys.modules:
    zip_path = Path.cwd() / "NLP_Inter_IIT_3.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for item in ["data", "predictions", "work", "check_format.py", "results_template.json"]:
            p = ROOT / item
            if not p.exists():
                continue
            for f in ([p] if p.is_file() else sorted(p.rglob("*"))):
                if f.is_file():
                    z.write(f, Path("NLP_Inter_IIT") / f.relative_to(ROOT))
    print(zip_path.name, f"{zip_path.stat().st_size / 1e6:.0f} MB")
    from google.colab import files
    files.download(str(zip_path))

NLP_Inter_IIT_3.zip 61 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>